<a href="https://colab.research.google.com/github/JONNY-ME/multilingual-health-qa/blob/main/01_gemma4_e4b_zero_shot_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 E4B-it Zero-Shot Baseline

This notebook runs an end-to-end zero-shot baseline for the Zindi **Multilingual Health Question Answering in Low-Resource African Languages Challenge** using `google/gemma-4-E4B-it`.

The workflow is intentionally baseline-only:

- no fine-tuning
- no external data
- deterministic generation by default
- local ROUGE-1 and ROUGE-L validation
- Zindi submission CSV generation

Recommended Colab runtime: **Colab Pro+ GPU** with A100, L4, or another high-memory GPU.

## 1. Colab Runtime Setup

Run this notebook in a GPU runtime:

1. In Colab, select **Runtime > Change runtime type > GPU**.
2. Prefer A100 or L4 on Colab Pro+.
3. If `google/gemma-4-E4B-it` requires access in your session, create a Hugging Face token and run the login cell below.

The notebook installs packages with `uv` for reproducibility and speed.

In [1]:
! pip -q install uv
!uv pip install --system -q --upgrade "git+https://github.com/huggingface/transformers.git" accelerate torch pandas numpy tqdm rouge-score sentencepiece bitsandbytes huggingface_hub gdown safetensors torchvision

import transformers
print("Transformers version:", transformers.__version__)
print("Setup complete. If you previously imported transformers in this runtime, restart the session and run from the top.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 118.0 MB/s eta 0:00:00
Transformers version: 5.8.0.dev0
Setup complete. If you previously imported transformers in this runtime, restart the session and run from the top.


In [1]:
import gc
import json
import os
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from rouge_score import rouge_scorer
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 240)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


## 2. Data Download and Paths

The challenge files are available in the shared Google Drive folder:

`https://drive.google.com/drive/folders/1gqWeNhgZGqqJtemvn9a1pMuEjVmi2s0t?usp=sharing`

In Colab, run the next cell to download the CSV files into `/content/multilingual-health-question-answering/data`. If you already mounted Google Drive or uploaded the repository manually, you can skip the download cell and point `PROJECT_DIR` to the folder containing `data/`.

Expected files:

- `data/Train.csv`
- `data/Val.csv`
- `data/Test.csv`
- `data/SampleSubmission.csv`

In [2]:
# Download the official challenge CSV files from the shared Google Drive folder into Colab.
# Skip this cell if the files already exist under PROJECT_DIR/data.
import shutil

import gdown

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1gqWeNhgZGqqJtemvn9a1pMuEjVmi2s0t?usp=sharing"
COLAB_PROJECT_DIR = Path("/content/multilingual-health-question-answering")
COLAB_DATA_DIR = COLAB_PROJECT_DIR / "data"
EXPECTED_DATA_FILES = ["Train.csv", "Val.csv", "Test.csv", "SampleSubmission.csv"]

COLAB_DATA_DIR.mkdir(parents=True, exist_ok=True)

missing_before = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_before:
    print("Downloading missing data files:", missing_before)
    gdown.download_folder(
        url=DRIVE_FOLDER_URL,
        output=str(COLAB_DATA_DIR),
        quiet=False,
        use_cookies=False,
    )
else:
    print("All expected data files already exist. Skipping download.")

# gdown may preserve a nested Drive folder name. Normalize files into COLAB_DATA_DIR.
for filename in EXPECTED_DATA_FILES:
    target = COLAB_DATA_DIR / filename
    if target.exists():
        continue
    matches = [path for path in COLAB_PROJECT_DIR.rglob(filename) if path.is_file()]
    if matches:
        shutil.copy2(matches[0], target)

missing_after = [name for name in EXPECTED_DATA_FILES if not (COLAB_DATA_DIR / name).exists()]
if missing_after:
    raise FileNotFoundError(f"Could not find expected data files after download: {missing_after}")

print("Data files ready:")
for filename in EXPECTED_DATA_FILES:
    path = COLAB_DATA_DIR / filename
    print(f"- {path} ({path.stat().st_size / 1024**2:.2f} MB)")

All expected data files already exist. Skipping download.
Data files ready:
- /content/multilingual-health-question-answering/data/Train.csv (18.27 MB)
- /content/multilingual-health-question-answering/data/Val.csv (4.26 MB)
- /content/multilingual-health-question-answering/data/Test.csv (0.34 MB)
- /content/multilingual-health-question-answering/data/SampleSubmission.csv (0.24 MB)


In [3]:
# Optional: mount Google Drive in Colab.
# from google.colab import drive
# drive.mount('/content/drive')

# Update this path in Colab if your project is stored elsewhere.
PROJECT_DIR_CANDIDATES = [
    Path("/content/multilingual-health-question-answering"),
    Path("/content/drive/MyDrive/multilingual-health-question-answering"),
    Path.cwd(),
]

PROJECT_DIR = None
for candidate in PROJECT_DIR_CANDIDATES:
    if (candidate / "data" / "Train.csv").exists():
        PROJECT_DIR = candidate
        break

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find data/Train.csv. Set PROJECT_DIR manually to the project folder."
    )

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "Train.csv"
VAL_PATH = DATA_DIR / "Val.csv"
TEST_PATH = DATA_DIR / "Test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "SampleSubmission.csv"

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_DIR: /content/multilingual-health-question-answering
DATA_DIR: /content/multilingual-health-question-answering/data
OUTPUT_DIR: /content/multilingual-health-question-answering/outputs


In [4]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_sub_df = pd.read_csv(SAMPLE_SUB_PATH)

REQUIRED_COLUMNS = {
    "train": {"ID", "input", "output", "subset"},
    "val": {"ID", "input", "output", "subset"},
    "test": {"ID", "input", "subset"},
    "sample_submission": {"ID", "TargetRLF1", "TargetR1F1", "TargetLLM"},
}

for name, df in {
    "train": train_df,
    "val": val_df,
    "test": test_df,
    "sample_submission": sample_sub_df,
}.items():
    missing = REQUIRED_COLUMNS[name] - set(df.columns)
    if missing:
        raise ValueError(f"{name} is missing required columns: {sorted(missing)}")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_sub_df.shape)

assert test_df["ID"].tolist() == sample_sub_df["ID"].tolist(), "Test IDs do not match sample submission order."
print("Sample submission ID order matches Test.csv.")

Train: (29815, 4)
Validation: (6686, 4)
Test: (2618, 3)
Sample submission: (2618, 4)
Sample submission ID order matches Test.csv.


In [5]:
def summarize_frame(name: str, df: pd.DataFrame) -> None:
    print(f"\n{name}")
    print("rows:", len(df))
    print("duplicate IDs:", df["ID"].duplicated().sum())
    print("missing values:")
    print(df.isna().sum())
    print("subset distribution:")
    print(df["subset"].value_counts().sort_index())


def add_length_columns(df: pd.DataFrame, text_col: str, prefix: str) -> pd.DataFrame:
    out = df.copy()
    out[f"{prefix}_words"] = out[text_col].fillna("").astype(str).str.split().str.len()
    out[f"{prefix}_chars"] = out[text_col].fillna("").astype(str).str.len()
    return out

summarize_frame("Train", train_df)
summarize_frame("Validation", val_df)
summarize_frame("Test", test_df)

train_lens = add_length_columns(train_df, "output", "output")
val_lens = add_length_columns(val_df, "output", "output")
length_summary = pd.DataFrame({
    "split": ["train", "validation"],
    "output_words_mean": [train_lens["output_words"].mean(), val_lens["output_words"].mean()],
    "output_words_median": [train_lens["output_words"].median(), val_lens["output_words"].median()],
    "output_words_p95": [train_lens["output_words"].quantile(0.95), val_lens["output_words"].quantile(0.95)],
    "output_chars_mean": [train_lens["output_chars"].mean(), val_lens["output_chars"].mean()],
})
display(length_summary.round(2))


Train
rows: 29815
duplicate IDs: 0
missing values:
ID        0
input     0
output    0
subset    0
dtype: int64
subset distribution:
subset
Aka_Gha    4455
Amh_Eth    1845
Eng_Eth    3915
Eng_Gha    4443
Eng_Ken    2080
Eng_Uga    7624
Lug_Uga    3383
Swa_Ken    2070
Name: count, dtype: int64

Validation
rows: 6686
duplicate IDs: 0
missing values:
ID        0
input     0
output    0
subset    0
dtype: int64
subset distribution:
subset
Aka_Gha    1114
Amh_Eth     462
Eng_Eth     564
Eng_Gha    1104
Eng_Ken     390
Eng_Uga    1688
Lug_Uga     846
Swa_Ken     518
Name: count, dtype: int64

Test
rows: 2618
duplicate IDs: 0
missing values:
ID        0
input     0
subset    0
dtype: int64
subset distribution:
subset
Aka_Gha    492
Amh_Eth     61
Eng_Eth     60
Eng_Gha    491
Eng_Ken    167
Eng_Uga    744
Lug_Uga    374
Swa_Ken    229
Name: count, dtype: int64


,split,output_words_mean,output_words_median,output_words_p95,output_chars_mean
0,train,76.22,61.0,184.0,492.85
1,validation,79.26,66.0,191.0,513.01


In [6]:
# Show one example per subset from validation.
example_cols = ["ID", "subset", "input", "output"]
examples = (
    val_df.sort_values("ID")
    .groupby("subset", group_keys=False)
    .head(1)[example_cols]
    .sort_values("subset")
)
display(examples)

,ID,subset,input,output
788,ID_VL_Aka_Gha_001A9A8B,Aka_Gha,"Sɛ mihyia ɔhaw bi wɔ abusua nhyehyɛe kwan bi ho, te sɛ nea ɛma ɔhaw bi ba me nipadua so 'side effects' anaa sɛ ɛyɛ den sɛ mede bedi dwuma a, dɛn na ɛsɛ sɛ meyɛ?","Anamɔn a Ɛsɛ sɛ Wotu: Kɔ Akwahosan Ho Ɔhwɛfo nkyɛn: Sɛ wuhyia ɔhaw bi a, kɔ w’akwahosan ho ɔyaresafo nkyɛn. Susu anamɔn foforɔ ho: Sɛ ɔkwan biako mfata a, wo ne nea ɔde ma wo no nsusu akwan foforo a wobɛfa so ayɛ ho. Di Akwankyerɛ Akyi:..."
1509,ID_VL_Amh_Eth_004B6FAB,Amh_Eth,ከሄርፒስ ሙሉ በሙሉ መፈወስ ይችላልን?,አይቻልም፣ ነገር ግን የፀረ-ቫይረስ መድሃኒቶች ወረርሽኞችን እና ስርጭቱን መቀንስ ይችላሉ።
1590,ID_VL_Eng_Eth_00A81DE9,Eng_Eth,Is it true that can syphilis be prevented?,"This is a question about, Syphilis. Syphilis is cured with penicillin. If untreated, it damages the brain, heart, and body. Early testing and treatment prevent complications."
2408,ID_VL_Eng_Gha_0066853D,Eng_Gha,How can adolescents navigate the healthcare system to find affordable sexual and reproductive healthcare services that meet their needs and preferences?,"Adolescents can navigate the healthcare system to find affordable sexual and reproductive healthcare services by: Researching available resources, including community health centers, school-based clinics, family planning programs, and p..."
3385,ID_VL_Eng_Ken_00C59F30,Eng_Ken,Is it advisable to bring up any mental health issues during HIV appointments?,"Yes. Some HIV medications have side effects such as vivid dreams, sleeplessness, anxiety or depression. If you have any mental health issues your doctor may suggest a different treatment plan."
4434,ID_VL_Eng_Uga_002C4C22,Eng_Uga,"How would you define chlamydia?, please answer in detail.","Chlamydia or Chlamydial Genitourinary Infection is a common sexually transmitted infection that affects both men and women and is caused by the bacteria Chlamydia trachomatis. Chlamydia infections are treatable and curable. However, the..."
6040,ID_VL_Lug_Uga_00B3700F,Lug_Uga,Enzijanjaba y'Omusujja gw'ekibumba ogwa B n'ey'akawuka akaleeta siriimu y'emu?,"Nedda, Eddagala ly'Omusujja gw'ekibumba ogwa B n'ery'akawuka akaleeta siriimu si lye limu, ebifaanana mu bika by'eddagala ebimu."
6262,ID_VL_Swa_Ken_000B1F5E,Swa_Ken,"Je, mtu anapaswa kufanya nini ikiwa anashiriki ngono isiyo salama na mpenzi ambaye ana shaka hadhi yake?","Ikiwa umefanya ngono bila kinga na mwenzi wako na unashuku kuwa anaweza kuwa na Ukimwi au ana maambukizi yoyote ya zinaa (STI), ni muhimu kuchukua hatua za haraka na zinazowajibika ili kulinda afya na ustawi wako. Hizi ndizo hatua unazo..."


## 3. Model Access

If Hugging Face access is required in your Colab session, uncomment the login cell and paste a token with model read permissions.

The model is loaded in zero-shot inference mode only. No fine-tuning or adapters are used.

In [7]:
# Optional Hugging Face login. Run only if model download fails due to authentication.
# from huggingface_hub import notebook_login
# notebook_login()

In [8]:
MODEL_ID = "google/gemma-4-E4B-it"
USE_4BIT = False  # Set True if the Colab GPU cannot load the BF16/auto model.

print("MODEL_ID:", MODEL_ID)
print("USE_4BIT:", USE_4BIT)

MODEL_ID: google/gemma-4-E4B-it
USE_4BIT: False


In [9]:
import sys
import gc
import torch

try:
    from transformers import AutoConfig, AutoProcessor, BitsAndBytesConfig
except RuntimeError as e:
    if "torchvision" in str(e):
        raise RuntimeError("\n\n>>> IMPORTANT: You must RESTART THE RUNTIME (Runtime > Restart session) for the torchvision installation to take effect. Then run all cells from the top. <<<\n\n") from e
    raise

try:
    from transformers import AutoModelForImageTextToText
    MODEL_AUTO_CLASS = AutoModelForImageTextToText
except ImportError:
    # Fallback for older source builds. If this branch fails for Gemma 4, rerun the setup cell and restart runtime.
    from transformers import AutoModelForCausalLM
    MODEL_AUTO_CLASS = AutoModelForCausalLM


def clear_gpu_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


clear_gpu_memory()

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
print("Loaded config model_type:", config.model_type)
print("Using model class:", MODEL_AUTO_CLASS.__name__)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

model_kwargs = {
    "device_map": "auto",
    "torch_dtype": "auto",
    "trust_remote_code": True,
}

if USE_4BIT:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model = MODEL_AUTO_CLASS.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()

print("Model loaded.")
if torch.cuda.is_available():
    print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
    print("Reserved GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))


config.json: 0.00B [00:00, ?B/s]

Loaded config model_type: gemma4
Using model class: AutoModelForImageTextToText


processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded.
Allocated GB: 14.79
Reserved GB: 15.02


In [10]:
SUBSET_HINTS = {
    "Aka_Gha": "Akan/Twi as used in Ghana",
    "Lug_Uga": "Luganda as used in Uganda",
    "Swa_Ken": "Kiswahili as used in Kenya",
    "Amh_Eth": "Amharic as used in Ethiopia",
    "Eng_Uga": "English as used in Uganda",
    "Eng_Gha": "English as used in Ghana",
    "Eng_Ken": "English as used in Kenya",
    "Eng_Eth": "English as used in Ethiopia",
}

SYSTEM_PROMPT = """You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations."""


def build_messages(question: str, subset: str) -> list[dict[str, str]]:
    language_hint = SUBSET_HINTS.get(subset, subset)
    user_prompt = f"""Subset/language cue: {subset} ({language_hint})

Question:
{question}

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def render_prompt(question: str, subset: str) -> str:
    messages = build_messages(question, subset)
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

print(render_prompt(val_df.iloc[0]["input"], val_df.iloc[0]["subset"])[:1000])

<bos><|turn>system
You are a careful multilingual health question-answering assistant for maternal, sexual, and reproductive health.
Answer accurately, clearly, and respectfully.
Do not invent medical facts, drug dosages, or legal requirements.
If the question is in a non-English African language, answer in that same language.
Return only the final answer text. Do not add labels, markdown, explanations, translations, or citations.<turn|>
<|turn>user
Subset/language cue: Aka_Gha (Akan/Twi as used in Ghana)

Question:
Sɛn na nwomasua ne adwuma nteteeɛ boa akuo a eye mmabun a wɔ hia neaɛma sokoronko ne ohaw ahorow, atubrafo, anaa wɔn a wɔte beae a ntawantawa wɔ mu?

Write a helpful health answer in the same language as the question. Keep the answer concise but complete, similar to a curated health FAQ response. Return only the answer.<turn|>
<|turn>model



In [11]:
THINKING_PATTERNS = [
    r"<\|channel\>thought.*?(?=<\|channel\>|$)",
    r"<think>.*?</think>",
    r"<\|think\|>",
]

SPECIAL_TOKEN_PATTERN = re.compile(r"<\|[^>]+\|>|</?s>|<pad>|<bos>|<eos>")
ROLE_PREFIX_PATTERN = re.compile(
    r"^(assistant|model|answer|final answer|response)\s*[:\-]\s*",
    flags=re.IGNORECASE,
)


def clean_generation(text: str) -> str:
    if text is None:
        return ""
    cleaned = str(text)
    for pattern in THINKING_PATTERNS:
        cleaned = re.sub(pattern, " ", cleaned, flags=re.DOTALL | re.IGNORECASE)
    cleaned = SPECIAL_TOKEN_PATTERN.sub(" ", cleaned)
    cleaned = cleaned.replace("```", " ")
    cleaned = cleaned.strip()
    cleaned = ROLE_PREFIX_PATTERN.sub("", cleaned).strip()
    cleaned = re.sub(r"\s+", " ", cleaned)
    return cleaned.strip(' \"\'')


@torch.inference_mode()
def generate_answer(
    question: str,
    subset: str,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    repetition_penalty: float = 1.05,
) -> str:
    prompt = render_prompt(question, subset)
    inputs = processor(text=prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
        "pad_token_id": processor.tokenizer.eos_token_id,
    }
    if temperature and temperature > 0:
        generation_kwargs.update({
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
        })
    else:
        generation_kwargs["do_sample"] = False

    outputs = model.generate(**inputs, **generation_kwargs)
    raw = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    if hasattr(processor, "parse_response"):
        try:
            parsed = processor.parse_response(raw)
            if isinstance(parsed, dict):
                raw = parsed.get("text") or parsed.get("response") or raw
            elif isinstance(parsed, str):
                raw = parsed
        except Exception:
            pass
    return clean_generation(raw)


def preview_generation(row: pd.Series) -> None:
    print("ID:", row["ID"])
    print("subset:", row["subset"])
    print("question:", row["input"])
    answer = generate_answer(row["input"], row["subset"])
    print("\nprediction:\n", answer)
    if "output" in row:
        print("\nreference:\n", row["output"])

print("Generation utilities ready.")

Generation utilities ready.


In [12]:
def run_generation_with_checkpoints(
    df: pd.DataFrame,
    output_path: Path,
    id_col: str = "ID",
    text_col: str = "input",
    subset_col: str = "subset",
    limit: int | None = None,
    checkpoint_every: int = 25,
    max_new_tokens: int = 220,
    temperature: float = 0.0,
    top_p: float = 0.95,
    overwrite: bool = False,
) -> pd.DataFrame:
    """Generate predictions and save progress so Colab interruptions can resume."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    work_df = df.copy()
    if limit is not None:
        work_df = work_df.head(limit).copy()

    if output_path.exists() and not overwrite:
        existing = pd.read_csv(output_path)
        if {id_col, "prediction"}.issubset(existing.columns):
            pred_map = dict(zip(existing[id_col], existing["prediction"].fillna("")))
        else:
            pred_map = {}
    else:
        pred_map = {}

    rows = []
    start_time = time.time()
    for idx, row in tqdm(work_df.iterrows(), total=len(work_df)):
        row_id = row[id_col]
        if row_id in pred_map and str(pred_map[row_id]).strip():
            prediction = pred_map[row_id]
        else:
            prediction = generate_answer(
                question=row[text_col],
                subset=row[subset_col],
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
            )
            pred_map[row_id] = prediction

        rows.append({
            id_col: row_id,
            subset_col: row[subset_col],
            text_col: row[text_col],
            "prediction": prediction,
        })

        if (len(rows) % checkpoint_every == 0) or (len(rows) == len(work_df)):
            checkpoint_df = pd.DataFrame(rows)
            checkpoint_df.to_csv(output_path, index=False)

    elapsed = time.time() - start_time
    result_df = pd.DataFrame(rows)
    result_df.to_csv(output_path, index=False)
    print(f"Saved {len(result_df)} predictions to {output_path}")
    print(f"Elapsed: {elapsed / 60:.1f} minutes")
    return result_df

print("Checkpointed inference utility ready.")

Checkpointed inference utility ready.


## 4. Validation Scoring

The public challenge score is a weighted mean of ROUGE-1 F1, ROUGE-L F1, and LLM-as-a-Judge. This baseline computes the two deterministic ROUGE components locally and reports a transparent partial weighted score.

The LLM judge component is not estimated here to keep the baseline reproducible and zero-shot.

In [13]:
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)


def score_pair(reference: str, prediction: str) -> dict[str, float]:
    reference = "" if pd.isna(reference) else str(reference)
    prediction = "" if pd.isna(prediction) else str(prediction)
    scores = rouge.score(reference, prediction)
    return {
        "rouge1_f1": scores["rouge1"].fmeasure,
        "rougeL_f1": scores["rougeL"].fmeasure,
    }


def score_predictions(pred_df: pd.DataFrame, reference_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    merged = pred_df.merge(
        reference_df[["ID", "output"]],
        on="ID",
        how="left",
        validate="one_to_one",
    )
    score_rows = [score_pair(ref, pred) for ref, pred in zip(merged["output"], merged["prediction"])]
    scored = pd.concat([merged.reset_index(drop=True), pd.DataFrame(score_rows)], axis=1)
    scored["local_rouge_weighted"] = 0.37 * scored["rouge1_f1"] + 0.37 * scored["rougeL_f1"]

    summary = scored.groupby("subset").agg(
        rows=("ID", "count"),
        rouge1_f1=("rouge1_f1", "mean"),
        rougeL_f1=("rougeL_f1", "mean"),
        local_rouge_weighted=("local_rouge_weighted", "mean"),
        pred_words=("prediction", lambda s: np.mean([len(str(x).split()) for x in s])),
        ref_words=("output", lambda s: np.mean([len(str(x).split()) for x in s])),
    ).reset_index()

    overall = pd.DataFrame([{
        "subset": "OVERALL",
        "rows": len(scored),
        "rouge1_f1": scored["rouge1_f1"].mean(),
        "rougeL_f1": scored["rougeL_f1"].mean(),
        "local_rouge_weighted": scored["local_rouge_weighted"].mean(),
        "pred_words": np.mean([len(str(x).split()) for x in scored["prediction"]]),
        "ref_words": np.mean([len(str(x).split()) for x in scored["output"]]),
    }])
    summary = pd.concat([overall, summary], ignore_index=True)
    return scored, summary

print("Scoring utilities ready.")

Scoring utilities ready.


In [14]:
# Smoke test: one validation example per subset.
smoke_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(1)
    .sort_values("subset")
    .reset_index(drop=True)
)

SMOKE_PATH = OUTPUT_DIR / "gemma4_e4b_zero_shot_smoke_predictions.csv"
smoke_pred_df = run_generation_with_checkpoints(
    smoke_df,
    SMOKE_PATH,
    checkpoint_every=1,
    overwrite=True,
)

smoke_scored_df, smoke_summary_df = score_predictions(smoke_pred_df, smoke_df)
display(smoke_summary_df.round(4))
display(smoke_scored_df[["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1"]])

  0%|          | 0/8 [00:00<?, ?it/s]

Saved 8 predictions to /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_smoke_predictions.csv
Elapsed: 0.5 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,8,0.2600,0.1564,0.1541,89.125,97.375
1,Aka_Gha,1,0.2687,0.1791,0.1657,33.000,76.000
2,Amh_Eth,1,0.0000,0.0000,0.0000,50.000,17.000
3,Eng_Eth,1,0.3137,0.1961,0.1886,71.000,30.000
4,Eng_Gha,1,0.2956,0.1379,0.1604,152.000,49.000
5,Eng_Ken,1,0.4040,0.2727,0.2504,81.000,107.000
6,Eng_Uga,1,0.2986,0.1415,0.1628,162.000,337.000
7,Lug_Uga,1,0.1871,0.1583,0.1278,68.000,51.000
8,Swa_Ken,1,0.3122,0.1659,0.1769,96.000,112.000


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1
0,ID_VL_Aka_Gha_EB2263DD,Aka_Gha,Ɔkwan bɛn so na mmabun betumi akamfo wɔn ho ahwɛ ahu sɛ wobenya ayaresa a obu ne animhwɛ nnim afi akwaahosan ho adwumayɛfo hɔ?,"Ɛsɛ sɛ mmabun di kan kyerɛkyerɛ wɔn ho akwannya ahorow ho asɛm, na wɔkyerɛ hokwan a wɔwɔ sɛ wɔde obu ne animhwɛ nnim di dwuma denam wɔn ahiade ne nea wɔpɛ a wɔbɛka akyerɛ akwahosan ho adwumayɛfo pefee no so. Wobetumi ahwehwɛ akwahosan h...","Mmabun kɛseɛ no biaraa ɛwɔ ɔyɛsɛm a ɛfa nnim afi akwaahosan ho adwumayɛfo hɔ, ɛwɔ ɛnyɛɛ sɛ ɛyɛ yɛn nti anigyeɛ. Ɛyɛ sɛ ɛyɛ sɛ yɛn nti anigyeɛ. Ɛyɛ sɛ yɛn nti anigyeɛ.<turn|>",0.268657,0.179104
1,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,የሰውነትዎ እና የውበትዎ እሴት ከውጫዊ ገጽታ በላይ ነው። እራስዎን መቀበል እና በራስ መተማመን ማዳበር በጣም አስፈላጊ የጤና አካል ነው። ሁሉም ሰው ልዩ እና ውድ ነው። ለጤናዎ እና ደህንነትዎ ትኩረት መስጠት፣ የአኗኗር ዘይቤዎን መንከባከብ እና በአዎንታዊ መልኩ ስለራስዎ ማሰብ ይጀምሩ። አስፈላጊ ከሆነ፣ ስሜቶችዎን ለመወያየት ከጤና ባለሙያ ጋር መነጋገር ጠቃሚ ሊሆን ይች...,0.000000,0.000000
2,ID_VL_Eng_Eth_01902620,Eng_Eth,"If I get the medicine for Gonorrhea, should my partner also take the medicine, even if he says he feels fine?","This is a question about, Partner Treatment. Yes, always. He may be asymptomatic (no signs), but he will still reinfect you. Both partners must be treated at the same time.","Yes, it is strongly recommended that your partner also be tested and treated for Gonorrhea, even if they have no symptoms. Gonorrhea can often be asymptomatic in partners, meaning they feel fine but can still pass the infection to you a...",0.313725,0.196078
3,ID_VL_Eng_Gha_2CD2A404,Eng_Gha,"Are there specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, and how can adolescents find them?","There are specific healthcare providers or clinics that specialize in providing medical assistance to adolescents in crisis situations, such as sexual health clinics, family planning clinics, or adolescent health centers. Adolescents ca...","Yes, there are specialized services available. Adolescents in crisis situations can seek help from several types of providers: 1. **Adolescent-Friendly Health Services (AFHS):** Many public and private health facilities now have dedicat...",0.295567,0.137931
4,ID_VL_Eng_Ken_0B531932,Eng_Ken,Can I have a child with a HIV patient without getting infected?,"Yes, it is possible for an HIV-negative individual to have a child with an HIV-positive partner without acquiring HIV if certain precautions and medical interventions are followed. The process typically involves a combination of strateg...","Yes, it is possible to have a child with an HIV-positive partner without getting infected, provided that the mother is on effective Antiretroviral Therapy (ART) throughout pregnancy, labour, and breastfeeding, and that appropriate preca...",0.404040,0.272727
5,ID_VL_Eng_Uga_629EF3D8,Eng_Uga,"How can I protect myself from contracting Herpes Simplex Virus?, please answer in detail.","The only way to completely avoid STIs is to not have vaginal, anal, or oral sex. If you are sexually active, you can do the following things to lower your chances of getting Genital Herpes or Herpes Simplex Virus (HSV), also known as he...","To protect yourself from contracting Herpes Simplex Virus (HSV), which can be transmitted through skin-to-skin contact, especially during sexual activity, a multi-faceted approach is recommended: **1. Safer Sexual Practices:** * **Condo...",0.298625,0.141454
6,ID_VL_Lug_Uga_AD409244,Lug_Uga,Nsobola ntya okuwangaala n'Akawuka akaleeta Siriimu/Siriimu?,"Okuwangaala n'okuweeweeza Akawuka akaleeta Siriimu kisobokera ddala n'obujjanjabi obutuufu, obujjanjabi n'engeri gy'osalawo okweyisa. Abantu abalina Akawuka akaleeta Siriimu basobola okuwangaala nako mu bulamu obweyagaza nga balina obuj...","Okukwangaala n'Akawuka akaleeta Siriimu (Syphilis) kyechaka mu ngeri zinyiira, kubanga ko si bya

In [15]:
# Stratified validation sample. Increase SAMPLE_PER_SUBSET after the smoke test looks healthy.
SAMPLE_PER_SUBSET = 10
sample_val_df = (
    val_df.sample(frac=1.0, random_state=SEED)
    .groupby("subset", group_keys=False)
    .head(SAMPLE_PER_SUBSET)
    .sort_values(["subset", "ID"])
    .reset_index(drop=True)
)

SAMPLE_VAL_PATH = OUTPUT_DIR / f"gemma4_e4b_zero_shot_val_sample_{SAMPLE_PER_SUBSET}_per_subset.csv"
sample_pred_df = run_generation_with_checkpoints(
    sample_val_df,
    SAMPLE_VAL_PATH,
    checkpoint_every=10,
    overwrite=False,
)

sample_scored_df, sample_summary_df = score_predictions(sample_pred_df, sample_val_df)
display(sample_summary_df.round(4))

  0%|          | 0/80 [00:00<?, ?it/s]

Saved 80 predictions to /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_val_sample_10_per_subset.csv
Elapsed: 4.6 minutes


,subset,rows,rouge1_f1,rougeL_f1,local_rouge_weighted,pred_words,ref_words
0,OVERALL,80,0.2567,0.1660,0.1564,78.1,80.175
1,Aka_Gha,10,0.2002,0.1407,0.1261,69.9,97.500
2,Amh_Eth,10,0.1022,0.1022,0.0756,43.0,15.800
3,Eng_Eth,10,0.2770,0.1835,0.1704,60.4,23.600
4,Eng_Gha,10,0.4157,0.2192,0.2349,132.9,84.500
5,Eng_Ken,10,0.3258,0.2223,0.2028,74.4,91.500
6,Eng_Uga,10,0.3064,0.1758,0.1784,111.1,146.400
7,Lug_Uga,10,0.1356,0.1088,0.0904,49.5,78.700
8,Swa_Ken,10,0.2909,0.1754,0.1725,83.6,103.400


In [16]:
# Full validation can take a long time. Set RUN_FULL_VALIDATION = True after sample validation passes.
RUN_FULL_VALIDATION = False

FULL_VAL_PATH = OUTPUT_DIR / "gemma4_e4b_zero_shot_val_predictions.csv"
FULL_VAL_SCORED_PATH = OUTPUT_DIR / "gemma4_e4b_zero_shot_val_scored.csv"
FULL_VAL_SUMMARY_PATH = OUTPUT_DIR / "gemma4_e4b_zero_shot_val_summary.csv"

if RUN_FULL_VALIDATION:
    full_val_pred_df = run_generation_with_checkpoints(
        val_df,
        FULL_VAL_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
    full_val_scored_df, full_val_summary_df = score_predictions(full_val_pred_df, val_df)
    full_val_scored_df.to_csv(FULL_VAL_SCORED_PATH, index=False)
    full_val_summary_df.to_csv(FULL_VAL_SUMMARY_PATH, index=False)
    display(full_val_summary_df.round(4))
else:
    print("RUN_FULL_VALIDATION is False. Using sample_scored_df for downstream error analysis.")
    full_val_scored_df = sample_scored_df.copy()
    full_val_summary_df = sample_summary_df.copy()

RUN_FULL_VALIDATION is False. Using sample_scored_df for downstream error analysis.


In [17]:
def add_error_flags(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()
    out["prediction"] = out["prediction"].fillna("").astype(str)
    out["pred_words"] = out["prediction"].str.split().str.len()
    out["ref_words"] = out["output"].fillna("").astype(str).str.split().str.len()
    out["is_empty"] = out["prediction"].str.strip().eq("")
    out["maybe_too_short"] = out["pred_words"] < 0.35 * out["ref_words"].clip(lower=1)
    out["maybe_too_long"] = out["pred_words"] > 2.5 * out["ref_words"].clip(lower=1)
    out["contains_refusal"] = out["prediction"].str.contains(
        r"cannot|can't|unable|consult a doctor|medical professional|as an ai",
        case=False,
        regex=True,
    )
    return out

analysis_df = add_error_flags(full_val_scored_df)

print("Potential output issues:")
display(analysis_df[["is_empty", "maybe_too_short", "maybe_too_long", "contains_refusal"]].mean().to_frame("rate"))

print("Lowest ROUGE-L examples:")
lowest_cols = ["ID", "subset", "input", "output", "prediction", "rouge1_f1", "rougeL_f1", "pred_words", "ref_words"]
display(analysis_df.sort_values("rougeL_f1").head(10)[lowest_cols])

print("Best ROUGE-L examples:")
display(analysis_df.sort_values("rougeL_f1", ascending=False).head(5)[lowest_cols])

Potential output issues:


,rate
is_empty,0.0000
maybe_too_short,0.0625
maybe_too_long,0.2125
contains_refusal,0.0125


Lowest ROUGE-L examples:


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1,pred_words,ref_words
12,ID_VL_Amh_Eth_4DD8EB85,Amh_Eth,ገና 14 ዓመቴ ነው፣ እና ክሊኒኩ ትልቅ ሰው ማምጣት እንዳለብኝ ነገረኝ። አልችልም፣ ምን ማድረግ ይገባኛል?,ማህበራዊ ሰራተኛ ወይም ልዩ የጤና ሰራተኛ ይጠይቁ። ለአካለ መጠን ያልደረሱ ልጆች ያለ ቤተሰባቸው ሚስጥራዊ እንክብካቤ እንዲያገኙ ለመርዳት የሰለጠኑ ናቸው።,"የጤና ባለሙያዎ የተወሰነ ምርመራ ወይም ሂደት እንዲደረግልዎ ጠይቀዋል ማለት ነው። ይህንን በተመለከተ የሚከተሉትን ማድረግ ይችላሉ፦ 1. **ጥያቄዎችን ይጠይቁ:** ለምን ይህ ምርመራ እንደሚያስፈልግ፣ ምን አይነት ምርመራ እንደሆነ፣ እና ለምን አሁን እንደሚያስፈልግ በዝርዝር ይጠይቁ። 2. **ጭንቀትዎን ይግለጹ:** ""አልችልም"" ብለው ከተናገሩ፣ የትኛው ክፍል ነው የሚያስቸግ...",0.000000,0.000000,75,19
13,ID_VL_Amh_Eth_670ACC5C,Amh_Eth,ለተጨማሪ ደህንነት ሁለት ኮንዶም በአንድ ጊዜ (አንዱ በሌላው ላይ) መጠቀም ደህንነቱ የተጠበቀ ነው?,አይሁለት ኮንዶም መጠቀም አንዱን ከመጠቀም ያነሰ ደህንነቱ የተጠበቀ ነው፣ ምክንያቱም በንብርብሮች መካከል ያለው ግጭት ሁለቱንም በቀላሉ እንዲቀደዱ ሊያደርግ ይችላል። ሁል ጊዜ ነጠላ ኮንዶም በትክክል ይጠቀሙ።,አንድ ኮንዶም በሌላኛው ላይ መጠቀም ተጨማሪ የደህንነት ሽፋን ይሰጣል፣ ነገር ግን ዋናው ጥበቃ የሚመጣው ትክክለኛውን እና ያልተሰበረውን ኮንዶም በአግባቡ በመጠቀም ነው። ሁለቱን በአንድ ላይ መጠቀም አደጋውን በእጅጉ ሊቀንስ ይችላል።<turn|>,0.000000,0.000000,29,25
15,ID_VL_Amh_Eth_7A8D03AA,Amh_Eth,ሄርፒስ እንዴት ይተላለፋል?,ሄርፒስ ምንም አይነት ምልክት ባይኖርዉም እንኳ የሚተላለፈው ቆዳ ለቆዳ በመነካካት፣ በመሳሳም ወይም በጾታዊ ግንኙነት ነው።,ሄርፒስ በዋናነት በሰውነት ንክኪ አማካኝነት ይተላለፋል። በተለይም በጾታዊ ግንኙነት ወቅት ፈሳሽ ወይም ቁስለት ሲኖር በቀላሉ ሊተላለፍ ይችላል። ከሰው ወደ ሰው የሚተላለፈው የቫይረሱ መገኘት ነው።<turn|>,0.000000,0.000000,24,15
11,ID_VL_Amh_Eth_409D3602,Amh_Eth,በፕሮቶዞአን ጥገኛ ተውሳክ የሚከሰተው የትኛው ዓይነት የአባላዘር በሽታ ነው?,ትሪኮሞኒየስ (Trichomonas vaginalis)፡፡,በፕሮቶዞአን ጥገኛ ተውሳክ የሚከሰተው የባህሪ በሽታዎች (Protozoan infections) ነው።<turn|>,0.000000,0.000000,9,3
19,ID_VL_Amh_Eth_B04D3376,Amh_Eth,አንድ ሕፃን ከተወለደ በኋላ ወዲያውኑ የዓይን ሕመም ቢይዝ ይህ ሕመም ከእናቱ የተላለፈ ሊሆን ይችላል?,አዎ፣ ሊሆን ይችላል። በሕፃናት ላይ የሚከሰት የአይን ኢንፌክሽኖች (neonatal conjunctivitis)በዋነኝነት ምክንያቱ በወሊድ ጊዜ በሚተላለፍ ክላሚዲያ ወይም ጨብጥ ነው። ዓይነ ስውርነትን ለመከላከል ወዲያውኑ መታከም አለበት።,የ新生儿的眼睛疼痛可能由多种原因引起，不一定都是由母亲传染的。一些常见的原因包括： * **新生儿感染：** 某些细菌或病毒感染可能导致眼部问题。 * **生理性刺激：** 新生儿的眼睛对光线、环境变化等非常敏感，可能会出现不适感。 * **先天性问题：** 有些眼部结构或发育上的问题也可能引起疼痛或不适。 * **母体因素（间接）：** 如果母亲在怀孕期间或分娩过程中有未被控制的感染（如某些性传播感染），理论上存在通过胎盘或分娩过程将某些病原体传递给婴儿的风险，但这...,0.000000,0.000000,15,24
18,ID_VL_Amh_Eth_AB8DDEB4,Amh_Eth,በጤና ጣቢያው ውስጥ ለነፍሰ ጡር ታዳጊ ልጃገረዶች ምን ዓይነት አገልግሎቶች ነፃ ናቸው?,የቅድመ ወሊድ እንክብካቤ ምርመራዎች፣ የቴታነስ ክትባቶች፣ የብረት/ፎሊክ አሲድ ተጨማሪዎች እና ልጅ መውለድ ብዙ ጊዜ ነፃ ወይም ዝቅተኛ ዋጋ አላቸው። ለበለጠ መረጃ የአካባቢዎን ክሊኒክ ያነጋግሩ።,በጤና ጣቢያዎች ውስጥ ለነፍሰ ጡር ታዳጊ ልጃገረዶች የሚሰጡ ነጻ አገልግሎቶች የሚከተሉትን ሊያካትቱ ይችላሉ፦ * **የፅንስ ክትትል (Antenatal Care - ANC):** መደበኛ የጤና ምርመራዎች፣ የደም ግፊት እና የደም መጠን መከታተል፣ እና የፅንሱ እድገት መከታተል። * **የክትባት:** ለእናቶች እና ለህፃናት አስፈላጊ የሆኑ ክትባቶች። * **የምክር አገልግሎት:** ...,0.000000,0.000000,76,24
17,ID_VL_Amh_Eth_A5CB63A2,Amh_Eth,በሰውነቴ እና በመልክዬ ኩራት ይሰማኛል?,አይከንቱ ወይም ጉራ መሆን ስህተት ነው፣ ነገር ግን የሰውነትዎን ጥንካሬ እና ጤንነት ማድነቅ ለእግዚአብሔር ፍጥረት አክብሮት ያሳያል።,የሰውነትዎ እና የውበትዎ እሴት ከውጫዊ ገጽታ በላይ ነው። እራስዎን መቀበል እና በራስ መተማመን ማዳበር በጣም አስፈላጊ የጤና አካል ነው። ሁሉም ሰው ልዩ እና ውድ ነው። ለጤናዎ እና ደህንነትዎ ትኩረት መስጠት፣ የአኗኗር ዘይቤዎን መንከባከብ እና በአዎንታዊ መልኩ ስለራስዎ ማሰብ ይጀምሩ። አስፈላጊ ከሆነ፣ ስሜቶችዎን ለመወያየት ከጤና ባለሙያ ጋር መነጋገር ጠቃሚ ሊሆን ይች...,0.000000,0.000000,50,17
16,ID_VL_Amh_Eth_9DAC6E83,Amh_Eth,ትሪኮሞኒያሲስ ለኤችአይቪ ተጋላጭነትን ይጨምራል?,አዎ። ትሪኮሞኒያሲስ ካለብዎት እና ለኤችአይቪ ከተጋለጡ በቫይረሱ የመያዝ ዕድልዎን ይበልጥ ከፍተኛ ያደርገዋል።,አዎ፣ ትሪኮሞኒያሲስ (Trichomoniasis) ለኤችአይቪ (HIV) ኢንፌክሽን ተጋላጭነትን ይጨምራል። ይህ የሆነበት ምክንያት የትሪኮሞኒያሲስ ኢንፌክሽን በሴት ብልት ውስጥ የኢሚዩኒቲ ስርዓትን (immune system) ሊዳከም ስለሚችል እና የግብረ ሥጋ ግንኙነት ወቅት የሴት ብልት መጎዳት (inflammation) ስለሚያመጣ ነው። እነዚህ ሁኔታዎች ኤችአይቪ ቫይረስ በቀላሉ ...,0.000000,0.000000,55,12
8,ID_VL_Aka_Gha_DEDA4E16,Aka_Gha,"Fa LGBTQ+ nneɛma a wɔahyehyɛ, a wɛbsaet, telefon a wɔde frɛ nkurɔfo, ne ɔman nnwumakuo ahorow ka ho ma.",wo sukuu ɔyarehwɛfo anaa akwahosan ho dwumayɛni ne wo bɛbɔ nna ho nsɛm ho nkɔmmɔ. Wobu LGBTQ+ sɛ mmara mma ho kwan wɔ Ghana,"LGBTQ+ nneɛma a wɔahyehyɛ, wɔwɔ sɛ wɔn de frɛ nkurɔfo, ne ɔman nnwumakuo ahorow ka ho ma, ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛyɛ anaa ɛy...",0.114754,0.049180,88,24
66,ID_VL_Lug_Uga_CD0579E2,Lug_Uga,Kisoboka Akawuka akaleeta Siriimu okumalibwawo ddala mu mubiri?,"Tewali ddagala liwonyeza ddala Kawuka akaleeeta Siriimu ng’oggyeeko eddagala erikaweweezaako. Enkola y'okuweweeza ku kawuka bwe 

Best ROUGE-L examples:


,ID,subset,input,output,prediction,rouge1_f1,rougeL_f1,pred_words,ref_words
14,ID_VL_Amh_Eth_6A5D932B,Amh_Eth,ትሪኮሞኒያሲስ የሚያመጣው ምን ዓይነት አካል ነው?,ነጠላ-ሕዋስ ፕሮቶዞአን ጥገኛ ተውሳክ (Trichomonas vaginalis)፡፡,ትሪኮሞኒያሲስ የሚከሰተው በሴት ብልት ውስጥ በሚገኝ ትሪኮሞናስ ቫጂናሊስ (Trichomonas vaginalis) የተባለ ጥገኛ ተውሳክ ነው።<turn|>,0.800000,0.800000,14,6
40,ID_VL_Eng_Ken_05D1FB63,Eng_Ken,Does sharing sex toys lead to the spread of STIs?,"Yes, the sharing of sex toys can potentially lead to the spread of sexually transmitted infections (STIs) among individuals, including girls. This is because sex toys can come into contact with bodily fluids, including vaginal or anal s...","Yes, sharing sex toys can potentially lead to the transmission of Sexually Transmitted Infections (STIs). STIs can be transmitted through bodily fluids present on the toys, such as semen, vaginal fluids, or blood. To prevent this risk, ...",0.504854,0.388350,60,42
38,ID_VL_Eng_Gha_C64C9647,Eng_Gha,What exactly does abstinence mean in terms of sexual health?,"Abstinence in terms of sexual health refers to refraining from engaging in any form of sexual activity that involves genital contact with a partner. This includes vaginal, anal, and oral intercourse, as well as other intimate activities...","Abstinence in sexual health means choosing not to engage in any sexual activity. This includes refraining from penile-vaginal intercourse, oral sex, anal sex, and any other form of sexual contact. The primary goal of abstinence is to pr...",0.526316,0.336842,48,45
21,ID_VL_Eng_Eth_336749B5,Eng_Eth,Can HPV affect men?,Yes. HPV affects both men and women and can cause cancers and genital warts in men.,"Yes, HPV can affect men. While it is often associated with cervical cancer in women, HPV can infect both men and women. In men, HPV can cause various issues, including genital warts and certain types of cancers, such as anal, penile, an...",0.410959,0.273973,56,16
34,ID_VL_Eng_Gha_8CCC9CEA,Eng_Gha,"What are the specific challenges faced by adolescents in conflict and post-conflict settings regarding their sexual and reproductive rights, autonomy, and well-being?","Violence and Exploitation: Adolescents in conflict and post-conflict settings face heightened risks of gender-based violence (GBV), sexual violence, early and forced marriage, and sexual exploitation, which pose significant threats to t...","Adolescents in conflict and post-conflict settings face numerous severe challenges concerning their sexual and reproductive rights, autonomy, and overall well-being. These challenges often stem from the instability, violence, and breakd...",0.444444,0.273504,165,67


## 5. Test Inference and Submission

Only run the full test generation after the smoke and validation sample outputs look reasonable. The generated answer must be duplicated exactly across `TargetRLF1`, `TargetR1F1`, and `TargetLLM`.

In [18]:
# Test generation can take time. Set RUN_TEST_INFERENCE = True when ready to produce a submission.
RUN_TEST_INFERENCE = True

TEST_PRED_PATH = OUTPUT_DIR / "gemma4_e4b_zero_shot_test_predictions.csv"
SUBMISSION_PATH = OUTPUT_DIR / "submission_gemma4_e4b_zero_shot.csv"

if RUN_TEST_INFERENCE:
    test_pred_df = run_generation_with_checkpoints(
        test_df,
        TEST_PRED_PATH,
        checkpoint_every=25,
        overwrite=False,
    )
else:
    print("RUN_TEST_INFERENCE is False.")
    if TEST_PRED_PATH.exists():
        print("Loading existing test predictions:", TEST_PRED_PATH)
        test_pred_df = pd.read_csv(TEST_PRED_PATH)
    else:
        print("No existing test predictions found. Run this cell with RUN_TEST_INFERENCE = True when ready.")
        test_pred_df = None

  0%|          | 0/2618 [00:00<?, ?it/s]

Saved 2618 predictions to /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_test_predictions.csv
Elapsed: 165.4 minutes


In [19]:
def create_submission(test_predictions: pd.DataFrame, sample_submission: pd.DataFrame, output_path: Path) -> pd.DataFrame:
    if test_predictions is None or test_predictions.empty:
        raise ValueError("test_predictions is empty. Run test inference first.")

    pred_cols = {"ID", "prediction"}
    missing = pred_cols - set(test_predictions.columns)
    if missing:
        raise ValueError(f"test_predictions is missing columns: {sorted(missing)}")

    pred_map = dict(zip(test_predictions["ID"], test_predictions["prediction"].fillna("")))
    missing_ids = [row_id for row_id in sample_submission["ID"] if row_id not in pred_map]
    if missing_ids:
        raise ValueError(f"Missing predictions for {len(missing_ids)} sample submission IDs. First few: {missing_ids[:5]}")

    answers = [clean_generation(pred_map[row_id]) for row_id in sample_submission["ID"]]
    if any(not ans for ans in answers):
        empty_count = sum(not ans for ans in answers)
        raise ValueError(f"Found {empty_count} empty predictions. Fix before submitting.")

    submission = pd.DataFrame({
        "ID": sample_submission["ID"],
        "TargetRLF1": answers,
        "TargetR1F1": answers,
        "TargetLLM": answers,
    })

    expected_cols = ["ID", "TargetRLF1", "TargetR1F1", "TargetLLM"]
    if submission.columns.tolist() != expected_cols:
        raise AssertionError("Submission columns are incorrect.")
    if len(submission) != len(sample_submission):
        raise AssertionError("Submission row count does not match sample submission.")
    if not submission["TargetRLF1"].equals(submission["TargetR1F1"]):
        raise AssertionError("TargetRLF1 and TargetR1F1 differ.")
    if not submission["TargetRLF1"].equals(submission["TargetLLM"]):
        raise AssertionError("TargetRLF1 and TargetLLM differ.")
    if submission.isna().any().any():
        raise AssertionError("Submission contains null values.")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    submission.to_csv(output_path, index=False)
    print("Saved submission:", output_path)
    print("Rows:", len(submission))
    return submission

if test_pred_df is not None:
    submission_df = create_submission(test_pred_df, sample_sub_df, SUBMISSION_PATH)
    display(submission_df.head())
else:
    print("Submission not created because test_pred_df is None.")

Saved submission: /content/multilingual-health-question-answering/outputs/submission_gemma4_e4b_zero_shot.csv
Rows: 2618


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"GBV (Gender-Based Violence) no akwatafoɔ kɛse no yɛ: 1. **Adwumayɛbea Ahorow (Physical Violence):** Ɛyɛ sɛnea ɔkɔnka, ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn agyeɛ biara. 2. **Akuo Ahorow (Sexual Violence):** Ɛyɛ sɛnea ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn...","GBV (Gender-Based Violence) no akwatafoɔ kɛse no yɛ: 1. **Adwumayɛbea Ahorow (Physical Violence):** Ɛyɛ sɛnea ɔkɔnka, ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn agyeɛ biara. 2. **Akuo Ahorow (Sexual Violence):** Ɛyɛ sɛnea ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn...","GBV (Gender-Based Violence) no akwatafoɔ kɛse no yɛ: 1. **Adwumayɛbea Ahorow (Physical Violence):** Ɛyɛ sɛnea ɔkɔnka, ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn agyeɛ biara. 2. **Akuo Ahorow (Sexual Violence):** Ɛyɛ sɛnea ɔkɔnfa, ɔkɔnpa, ne ɛnyɛɛ wɔn..."
1,ID_TS_Aka_Gha_1C80317F,"Nipafoɔ kɛseɛ a ɛbɛyɛ yɛn anaa wɔ nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu, ɛwɔ sɛnea: 1. **Mmoa Agyin (Infection):** Ahofadi no yɛ ɔkwan kɛseɛ sɛ ɛbɛyɛ mmoa agyin (infection), ɛwɔ sɛnea ɛyɛ mmoa agyin wɔ nipadua mu. 2...","Nipafoɔ kɛseɛ a ɛbɛyɛ yɛn anaa wɔ nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu, ɛwɔ sɛnea: 1. **Mmoa Agyin (Infection):** Ahofadi no yɛ ɔkwan kɛseɛ sɛ ɛbɛyɛ mmoa agyin (infection), ɛwɔ sɛnea ɛyɛ mmoa agyin wɔ nipadua mu. 2...","Nipafoɔ kɛseɛ a ɛbɛyɛ yɛn anaa wɔ nipadua mu ahofadi wɔ nna ne awoɔ akwahosan ho nhyehyɛe mu, ɛwɔ sɛnea: 1. **Mmoa Agyin (Infection):** Ahofadi no yɛ ɔkwan kɛseɛ sɛ ɛbɛyɛ mmoa agyin (infection), ɛwɔ sɛnea ɛyɛ mmoa agyin wɔ nipadua mu. 2..."
2,ID_TS_Aka_Gha_06671AD1,"Ɛyɛ yɛn yɛn sɛ yɛn deɛn no sɛ yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ den, yɛn nneɛma bɛn na yɛn bɛba mubi. Yɛn nneɛma bɛn na yɛn bɛba mubi sɛ yɛn bɛka nea yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ ...","Ɛyɛ yɛn yɛn sɛ yɛn deɛn no sɛ yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ den, yɛn nneɛma bɛn na yɛn bɛba mubi. Yɛn nneɛma bɛn na yɛn bɛba mubi sɛ yɛn bɛka nea yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ ...","Ɛyɛ yɛn yɛn sɛ yɛn deɛn no sɛ yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ den, yɛn nneɛma bɛn na yɛn bɛba mubi. Yɛn nneɛma bɛn na yɛn bɛba mubi sɛ yɛn bɛka nea yɛn ka nnipa bi a wɔdeɛ no sɛ 'bystander' wɔ asɛm bi mu ayɛ ..."
3,ID_TS_Aka_Gha_BDD640FB,"Ɛyɛ sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu. Ɛda ne sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu.<turn|>","Ɛyɛ sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu. Ɛda ne sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu.<turn|>","Ɛyɛ sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu. Ɛda ne sɛ yɛn deɛn no kyerɛw ne ho yɛn adwuma wɔ Maternal, Sexual, Reproductive Health (ASRH) mu.<turn|>"
4,ID_TS_Aka_Gha_46685257,Policy advocacy no so nea ɛwɔ sɛnea nnipa kɔmfoɔ wɔ biribiara mu wɔ yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn ...,Policy advocacy no so nea ɛwɔ sɛnea nnipa kɔmfoɔ wɔ biribiara mu wɔ yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn ...,Policy advocacy no so nea ɛwɔ sɛnea nnipa kɔmfoɔ wɔ biribiara mu wɔ yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn yɛn ...


In [20]:
from google.colab import files

if 'SUBMISSION_PATH' in locals() and SUBMISSION_PATH.exists():
    print(f"Downloading {SUBMISSION_PATH.name}...")
    files.download(str(SUBMISSION_PATH))
else:
    print("Submission file not found. Please ensure the submission generation cell ran successfully.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
print("Notebook complete. Main output files:")
print("Smoke predictions:", SMOKE_PATH)
print("Sample validation predictions:", SAMPLE_VAL_PATH)
print("Full validation predictions:", FULL_VAL_PATH)
print("Test predictions:", TEST_PRED_PATH)
print("Submission:", SUBMISSION_PATH)

Notebook complete. Main output files:
Smoke predictions: /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_smoke_predictions.csv
Sample validation predictions: /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_val_sample_10_per_subset.csv
Full validation predictions: /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_val_predictions.csv
Test predictions: /content/multilingual-health-question-answering/outputs/gemma4_e4b_zero_shot_test_predictions.csv
Submission: /content/multilingual-health-question-answering/outputs/submission_gemma4_e4b_zero_shot.csv


## Baseline Notes for Reporting

Record the following before submitting or comparing runs:

- Model: `google/gemma-4-E4B-it`
- Inference type: zero-shot, no fine-tuning
- Runtime: Colab Pro+ GPU type from `nvidia-smi`
- Decoding: deterministic by default, `temperature=0.0`
- Prompt: system prompt and user template defined above
- Validation scope: smoke, stratified sample, or full validation
- Known limitation: local score excludes the LLM-as-a-Judge component

Next recommended step after this baseline: compare against a retrieval baseline and then a QLoRA fine-tuned Gemma/African-language model.